In [3]:
import torch
import torch.nn as nn
import torchvision.models as models
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import pandas as pd
from PIL import Image
import os

# 1. Dataset Blueprint (Ensuring it's in memory)
class TrafficSignDataset(Dataset):
    def __init__(self, csv_file, transform=None):
        self.data_frame = pd.read_csv(csv_file)
        self.transform = transform

    def __len__(self):
        return len(self.data_frame)

    def __getitem__(self, idx):
        img_path = self.data_frame.iloc[idx]['Path']
        full_path = os.path.abspath(img_path)
        image = Image.open(full_path).convert('RGB')
        y_label = torch.tensor(int(self.data_frame.iloc[idx]['ClassId']))
        
        if self.transform:
            image = self.transform(image)
        return (image, y_label)

# 2. Setup Device
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 3. Build Model & Load Weights
print("Building ResNet18 and loading weights...")
model = models.resnet18() 
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 43) 
model = model.to(device)

model.load_state_dict(torch.load('resnet_traffic_sign.pth', map_location=device))
model.eval() # Turn off dropout/batchnorm for testing

# 4. Setup Test Data (224x224 + Normalization)
print("Loading Test data...")
transform = transforms.Compose([
    transforms.Resize((224, 224)), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_dataset = TrafficSignDataset(csv_file='Test.csv', transform=transform)
test_loader = DataLoader(dataset=test_dataset, batch_size=64, shuffle=False)

# 5. Run the Exam
correct = 0
total = 0

print("Grading all test images... (This will take a few seconds)")

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

# 6. Results
final_accuracy = 100 * correct / total
print("-" * 40)
print(f"Official ResNet18 Test Accuracy: {final_accuracy:.2f}%")
print(f"Got {correct} out of {total} images perfectly correct!")
print("-" * 40)

Using device: mps
Building ResNet18 and loading weights...
Loading Test data...
Grading all test images... (This will take a few seconds)
----------------------------------------
Official ResNet18 Test Accuracy: 64.53%
Got 8150 out of 12630 images perfectly correct!
----------------------------------------
